In [1]:
import tiktoken

In [2]:
text = "Hello! Byte Pair Encoding"
byte_arr = bytearray(text, "utf-8")
print(byte_arr)

bytearray(b'Hello! Byte Pair Encoding')


In [3]:
ids = list(byte_arr)
print(ids)

[72, 101, 108, 108, 111, 33, 32, 66, 121, 116, 101, 32, 80, 97, 105, 114, 32, 69, 110, 99, 111, 100, 105, 110, 103]


In [4]:
print("Number of characters", len(text))
print("Number of Bytes", len(ids))

Number of characters 25
Number of Bytes 25


Using Tiktoken

In [5]:
enc = tiktoken.encoding_for_model("gpt-4o")
tokens = enc.encode(text)
print(tokens)

[13225, 0, 20445, 41250, 70820]


Load Datatext

In [6]:
# I would be using the shakespeare text

def load_shakespeare():
    file_path = "/home/bukunmi/ml-journey/datasets/tinyshakespeare.txt"

    with open(file_path, "r", encoding="utf-8") as f:
        raw = f.read()
    return raw

In [7]:
text = load_shakespeare()
len(text)

1115394

In [58]:
from collections import Counter, defaultdict
import itertools, re

class BPETokenizer:
    def __init__(self, vocab_size = 50000):
        self.vocab_size = vocab_size
        self.vocab = {}
        self.merges = {}

    def _init_vocab(self, text):
        if not text.strip():
            return {}
        words = text.split()
        word_counts = Counter(words)
        bpe_vocab = {}
        for word, count in word_counts.items():
            char_tuple = tuple(list(word)) + ('</w>',)
            bpe_vocab[char_tuple] = count
        return bpe_vocab
        

    def _get_pair_frequencies(self, bpe_vocab):         
        pairs = defaultdict(int)
        for word_tuple, frequency in bpe_vocab.items():
            for pair in itertools.pairwise(word_tuple):
                pairs[pair] += frequency
        return pairs
    
    def _merge_vocab(self, pair, bpe_vocab):
        new_vocab = defaultdict(int)
        bigram = re.escape(pair[0]) + r'\s' + re.escape(pair[1])
        pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
        for word_tuple, frequency in bpe_vocab.items():
            word_str = ' '.join(word_tuple)
            merged_str = pattern.sub(''.join(pair), word_str)
            new_vocab[tuple(merged_str.split())] += frequency
        return dict(new_vocab)
    
    def fit(self, text):
        bpe_vocab = self._init_vocab(text)
        unique_chars = set(itertools.chain.from_iterable(bpe_vocab))
        current_vocab_size = len(unique_chars)
        self.merges = {}

        while current_vocab_size < self.vocab_size:
            pair_counts = self._get_pair_frequencies(bpe_vocab)
            if not pair_counts:
                break
            best_pair = max(pair_counts, key=lambda x: pair_counts[x])
            self.merges[best_pair] = "".join(best_pair)
            bpe_vocab = self._merge_vocab(best_pair, bpe_vocab)
            current_vocab_size += 1
        all_tokens = sorted(unique_chars) + list(self.merges.values())
        self.vocab = {token: idx for idx, token in enumerate(all_tokens)}
        return self
    
    def tokenize(self, text):
        words = text.split()
        tokenized_text = []
        for word in words:
            pass


In [59]:
from itertools import islice

In [62]:
tokenizer = BPETokenizer(1000)
word_list = tokenizer._init_vocab(text)
print(dict(islice(word_list.items(), 0, 4)))

{('F', 'i', 'r', 's', 't', '</w>'): 235, ('C', 'i', 't', 'i', 'z', 'e', 'n', ':', '</w>'): 98, ('B', 'e', 'f', 'o', 'r', 'e', '</w>'): 31, ('w', 'e', '</w>'): 658}


In [63]:
tokenizer.fit(text)